# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — access, connection, lane constants

The warehouse is on Hugging Face. DuckDB reads only the columns and month
partitions the SQL touches, so nothing here downloads all rows.

Every query below runs against the **mid-panel month
`2026-03`**.

In [8]:
import importlib.util, subprocess, sys

for pkg in ("duckdb", "huggingface_hub"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json, os, pathlib
import duckdb, numpy as np, pandas as pd

pd.set_option("display.width", 140)
SEED = 42  # every split/model below is seeded with this


def load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "environment variable"
    for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        env_file = parent / ".env"
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip().strip("\"'"), "local .env (gitignored)"
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN"), "Colab Secret"
    except Exception:
        pass
    raise RuntimeError("No HF_TOKEN found. Set a Colab Secret named HF_TOKEN (read token).")


HF_TOKEN, token_source = load_hf_token()
print(f"HF read token loaded from: {token_source}")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"


def fact(month):
    """One month partition of fact_content_daily_performance."""
    return f"read_parquet('{REL}/fact_content_daily_performance/month={month}/*.parquet')"


# --- lane constants: the whole contract in five numbers ---------------------
FEATURE_MONTH = "2026-03"          # mid-panel month; features are measured here
LABEL_MONTH   = "2026-04"          # strictly after the decision moment
DECISION_DATE = "2026-03-31"       # the moment the editor scores the queue
MIN_HISTORY   = "2026-01-01"       # client must have GSC history starting on/before this
MIN_IMP_MAR   = 100                # a page needs measurable March search demand
DECLINE_DROP  = 0.20               # "declining" = April impressions >20% below March
TOP_K         = 50                 # editorial capacity: the queue is 50 pages long

receipts = {}  # every number this notebook claims gets stored here and written to work/outputs/
print(f"decision moment {DECISION_DATE} | features from {FEATURE_MONTH} | label from {LABEL_MONTH}")

HF read token loaded from: environment variable
decision moment 2026-03-31 | features from 2026-03 | label from 2026-04


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My lane is the **Bleed Tracker**: rank top-tier pages by how likely they are to lose search
impressions next month, so an editor can work down a capped queue. Framed in w02 as a
scoring/ranking task — a classifier's predicted probability is the priority score.

Here are the **five contract answers in plain words**:

**A1 — One row = one page-month decision row.** One content item (`content_hash_id`) belonging
to one client (`client_hash_id`), summarized over the feature month **2026-03**, as it looked at
the decision moment **2026-03-31, end of day**. Not a page-day (the fact table's own grain), not
a page-lifetime: the editor re-scores the queue once a month, so the decision grain is the page
*per scoring month*. Within one scoring month `content_hash_id` alone identifies the row, and
`client_hash_id` rides along for grouped splits.

**A2 — Tables I use.** `fact_content_daily_performance` `month=2026-03` (all five features),
the same table's `month=2026-04` (the label), and `dim_clients` (per-client history depth, which
decides who is even eligible). `dim_content` is joined once, for a missingness check only — no
feature comes from it this week. Two tables I deliberately do **not** use:
`fact_content_daily_performance_sample` (it is the panel's last month, June 2026 — my sealed
test month) and `fact_content_query_90d` (its fixed 90-day window overlaps my label month, so
its `impressions_90d` / `*_last30` columns contain the outcome I am predicting).

**A3 — Time windows.** Feature window: **2026-03-01 → 2026-03-31** (31 days, entirely before the
decision moment). Label window: **2026-04-01 → 2026-04-30** (entirely after it). The two windows
touch but never overlap. My slice inside those windows: rows with `gsc_data_available IS TRUE`,
clients whose `gsc_data_start` is on or before 2026-01-01 (so a full, non-onboarding March
exists), and pages with at least 100 March impressions (below that, a "decline" is noise).

**A4 — What I predict/rank** (bucketed in section 2): Probability that April impressions fall more than 20% below
March impressions. Pages sort by that probability; the editor takes the top 50; the metric is
precision@50.

**A5 — What I deliberately exclude** (bucketed in section 2): anything measured in or after the label
window, every GA4 column, the query table's overlapping-window columns, and the hash ids as
model inputs.

The query below verifies A3's premise — that 2026-03 really is a **mid-panel** month with April
available after it, and that per-client history depth is uneven enough to need the
`gsc_data_start` filter at all.

In [9]:
# Does the panel actually support a March -> April window, and how uneven is client history?
panel = con.sql(f"""
    SELECT COUNT(*)                                                        AS clients_total,
           COUNT(*) FILTER (gsc_data_start IS NULL)                        AS gsc_start_missing,
           COUNT(*) FILTER (gsc_data_start <= DATE '{MIN_HISTORY}')        AS eligible_history,
           COUNT(*) FILTER (gsc_data_start >  DATE '{MIN_HISTORY}')        AS too_fresh,
           MIN(gsc_data_start)                                             AS earliest_start,
           MAX(gsc_data_start)                                             AS latest_start
    FROM {DIM_CLIENTS}
""").df()
print("dim_clients — per-client GSC history depth")
print(panel.to_string(index=False))

# Both months exist, and neither is the panel's final (sealed) month.
spans = con.sql(f"""
    SELECT '{FEATURE_MONTH}' AS month, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT report_date) AS days
    FROM {fact(FEATURE_MONTH)}
    UNION ALL
    SELECT '{LABEL_MONTH}', MIN(report_date), MAX(report_date), COUNT(DISTINCT report_date)
    FROM {fact(LABEL_MONTH)}
""").df()
print("\nfeature month and label month — dates present, no overlap")
print(spans.to_string(index=False))

receipts["clients_total"] = int(panel.clients_total[0])
receipts["clients_eligible_history"] = int(panel.eligible_history[0])
receipts["feature_window"] = [str(spans.first_day[0].date()), str(spans.last_day[0].date())]
receipts["label_window"] = [str(spans.first_day[1].date()), str(spans.last_day[1].date())]
assert spans.last_day[0] == pd.Timestamp(DECISION_DATE), "feature window must end at the decision moment"
assert spans.first_day[1] > spans.last_day[0], "label window must start after the feature window"
print("\nOK — feature window ends at the decision moment; label window starts strictly after it.")

dim_clients — per-client GSC history depth
 clients_total  gsc_start_missing  eligible_history  too_fresh earliest_start latest_start
           104                 37                40         27     2025-01-27   2026-06-02


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


feature month and label month — dates present, no overlap
  month  first_day   last_day  days
2026-03 2026-03-01 2026-03-31    31
2026-04 2026-04-01 2026-04-30    30

OK — feature window ends at the decision moment; label window starts strictly after it.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**A4 — the label.** `label_declining` = 1 when a page's April impressions are more than 20%
below its March impressions (`imp_apr < 0.8 × imp_mar`), else 0. It is computed from
`gsc_impressions` in `month=2026-04` — a **forward window that starts after the decision
moment**, which is the upgrade w02 promised: that notebook used the CSV's `is_declining_label`,
a rule applied to two windows that both sat inside the same elapsed 90-day snapshot. This is still a *proxy* for "the page is bleeding" (a 21% drop and a 19% drop are not
different events in the world), and `imp_mar` appears in both the label formula and my feature
list. That is allowed — March impressions are genuinely knowable on 2026-03-31 — but it means
the model can win partly by learning "small pages bounce around", so I check the honest score in
section 3 against the base rate rather than trusting it flat.

**Feature (5, all measured inside the feature window).** Every one is knowable at
2026-03-31 23:59, from `month=2026-03` rows with `gsc_data_available IS TRUE`:

| Feature | Built from | Knowable at the decision moment because… |
|---|---|---|
| `log_imp_mar` | `log1p(SUM(gsc_impressions))` | …it is March's own impression total, complete once March ends. |
| `active_days` | `COUNT(*) FILTER (gsc_impressions > 0)` | …it counts March days that already happened; nothing after 03-31 enters it. |
| `pos_mar` | `SUM(gsc_sum_position) / SUM(gsc_impressions)` | …it is the impression-weighted average rank *over March days only* (the sum-based form, not `AVG(gsc_avg_position)`, so zero-impression days can't drag it). |
| `ctr_mar` | `SUM(gsc_clicks) / SUM(gsc_impressions)` | …both numerator and denominator are March measurements, closed on 03-31. |
| `momentum_in_month` | March 17–31 impressions ÷ March 1–16 impressions | …it compares two halves of the *feature* month; it is a within-window trend, so it carries no April information. (The halves are 15 and 16 days, so a flat page scores just under 1 — observed median 0.99. The same tilt applies to every row, so it moves the scale, not the ranking.) |

**Context (grouping, joining, filtering, reading — never model inputs).**
`client_hash_id` (grouped train/test splits, per-client checks), `content_hash_id` (row
identity, joining March to April), `report_date` / `month` (windowing), `gsc_data_available`
(the availability filter), `dim_clients.gsc_data_start` (eligibility), `dim_content.content_type`
(missingness diagnosis in section 4), and `has_first_half_traffic` — the flag I build in section 3b to mark
pages whose `momentum_in_month` is undefined. It is knowable at the decision moment and would be
a legitimate sixth feature, but the contract caps this lane at five, so it stays context: built,
inspected, and kept out of the model.

**Excluded — with the why.**

| Excluded | Why |
|---|---|
| `gsc_impressions`/`gsc_clicks`/anything from `month=2026-04` (other than the label itself) | Future information: measured after the decision moment. This is exactly the column I inject on purpose in section 3 to show what leakage looks like. |
| every `ga4_*` column, `sessions_*`, `ai_*`, `scroll_events` | Availability is three-valued and patterned, not random: in March only ~4% of rows are `ga4_data_available IS TRUE` and ~31% are NULL. Rows before a client's `ga4_data_start` are zero-**filled**, so a zero means "not tracked", not "not engaged" — feeding those in teaches the model which clients bought analytics. |
| `fact_content_query_90d.*` (`impressions_90d`, `*_last30`, shares) | Its fixed 90-day window overlaps my April label window, so those columns contain the outcome. Only `*_prev30`-style columns could ever be safe here, and I don't need them for five features. |
| `client_hash_id`, `content_hash_id`, `url_hash_id`, `keyword_hash_id` as inputs | Pseudonyms with no meaning; as features they would just memorise which client a page belongs to and destroy cross-client generalisation. |
| `health_score`, `priority_score`, `action_type`, refresh flags | Product decisions — FlyRank's own rule outputs. Not shipped in this release, and if I rebuilt one it would be a baseline to beat, never a feature. |
| `trend_direction`, `trend_pct`, `is_declining_label` (starter CSV) | Label-derived by construction (w02's trap). They don't exist in the warehouse, and I do not re-import them. |

The cell below turns this table into a checked object: every field appears in exactly one
bucket, and nothing from the label or excluded buckets can reach the feature list.

In [10]:
BUCKETS = {
    "feature": ["log_imp_mar", "active_days", "pos_mar", "ctr_mar", "momentum_in_month"],
    "label":   ["imp_apr", "label_declining"],
    "context": ["client_hash_id", "content_hash_id", "report_date", "month",
                "gsc_data_available", "gsc_data_start", "content_type",
                "has_first_half_traffic"],
    "excluded": ["gsc_impressions@2026-04", "ga4_data_available", "ga4_sessions",
                 "ga4_engaged_sessions", "ga4_total_engagement_sec", "sessions_ai",
                 "scroll_events", "impressions_90d", "rare_impressions_share",
                 "url_hash_id", "keyword_hash_id", "health_score", "priority_score",
                 "action_type", "trend_direction", "trend_pct", "is_declining_label"],
}
WHY_EXCLUDED = {  # one line each, same reasons as the markdown table
    "gsc_impressions@2026-04": "future window — measured after the decision moment",
    "ga4_data_available": "three-valued availability flag; zero-filled rows are 'not tracked'",
    "ga4_sessions": "GA4 coverage is patterned by client, not random",
    "ga4_engaged_sessions": "GA4 coverage is patterned by client, not random",
    "ga4_total_engagement_sec": "GA4 coverage is patterned by client, not random",
    "sessions_ai": "GA4-derived; also far too sparse in this panel",
    "scroll_events": "GA4-derived; present on a small minority of rows",
    "impressions_90d": "query table's 90d window overlaps my April label window",
    "rare_impressions_share": "same overlapping window as above",
    "url_hash_id": "pseudonym — join/group only, never a model input",
    "keyword_hash_id": "pseudonym — join/group only, never a model input",
    "health_score": "product decision (rule output), not an observed signal",
    "priority_score": "product decision (rule output), not an observed signal",
    "action_type": "product decision (rule output), not an observed signal",
    "trend_direction": "label-derived by construction (w02 trap)",
    "trend_pct": "label-derived by construction (w02 trap)",
    "is_declining_label": "the old proxy label itself",
}

contract = pd.DataFrame(
    [{"field": f, "bucket": b, "why_excluded": WHY_EXCLUDED.get(f, "")}
     for b, fields in BUCKETS.items() for f in fields]
)
print(contract.to_string(index=False))

# Checks, not vibes: one bucket per field, a why for every exclusion, no leak path into features.
assert not contract.field.duplicated().any(), "a field lands in two buckets"
assert all(WHY_EXCLUDED.get(f) for f in BUCKETS["excluded"]), "every excluded field needs a why"
FEATURES = BUCKETS["feature"]
assert not set(FEATURES) & set(BUCKETS["label"] + BUCKETS["excluded"]), "leak path into FEATURES"
assert len(FEATURES) <= 5, "the contract caps this lane at five features"
receipts["n_features"] = len(FEATURES)
receipts["n_excluded"] = len(BUCKETS["excluded"])
print(f"\nOK — {len(contract)} fields bucketed, {len(FEATURES)} features, "
      f"{len(BUCKETS['excluded'])} exclusions each with a stated why.")

                   field   bucket                                                       why_excluded
             log_imp_mar  feature                                                                   
             active_days  feature                                                                   
                 pos_mar  feature                                                                   
                 ctr_mar  feature                                                                   
       momentum_in_month  feature                                                                   
                 imp_apr    label                                                                   
         label_declining    label                                                                   
          client_hash_id  context                                                                   
         content_hash_id  context                                                          

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three small queries against `month=2026-03`, in the order the contract claims them:

- **Q1 — grain.** A1 says one row per `report_date × client × content` in the raw fact, collapsing
  to one row per content item once I aggregate the month. The probe returns zero rows if that
  holds.
- **Q2 — counts and date span.** How big the month is, how big *my slice* is after each filter,
  and that the dates run 03-01 → 03-31.
- **Q3 — availability.** The flags are three-valued (TRUE / FALSE / NULL), so I filter with
  `IS TRUE` and show how many rows survive — `= TRUE` and `NOT ...` silently mishandle the NULLs.

In [11]:
# --- Q1: grain probe. Zero rows back = the stated grain holds. ---------------
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {fact(FEATURE_MONTH)}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Q1 — duplicate (date x client x content) rows found: {len(dupes)}  "
      f"(0 = the fact grain is what I said it is)")
receipts["q1_duplicate_grain_rows"] = int(len(dupes))
assert len(dupes) == 0

# --- Q2: counts + date span, raw month vs my slice ---------------------------
counts = con.sql(f"""
    SELECT COUNT(*)                          AS rows_in_month,
           COUNT(DISTINCT content_hash_id)   AS content_items,
           COUNT(DISTINCT client_hash_id)    AS clients,
           MIN(report_date)                  AS first_day,
           MAX(report_date)                  AS last_day
    FROM {fact(FEATURE_MONTH)}
""").df()
print("\nQ2 — the whole feature month")
print(counts.to_string(index=False))

slice_counts = con.sql(f"""
    WITH kept AS (
        SELECT f.client_hash_id, f.content_hash_id, SUM(f.gsc_impressions) AS imp_mar
        FROM {fact(FEATURE_MONTH)} f
        JOIN {DIM_CLIENTS} c USING (client_hash_id)
        WHERE f.gsc_data_available IS TRUE
          AND c.gsc_data_start <= DATE '{MIN_HISTORY}'
        GROUP BY 1, 2
    )
    SELECT COUNT(*)                                              AS pages_after_availability_and_history,
           COUNT(*) FILTER (imp_mar >= {MIN_IMP_MAR})            AS pages_in_my_slice,
           COUNT(DISTINCT client_hash_id) FILTER (imp_mar >= {MIN_IMP_MAR}) AS clients_in_my_slice
    FROM kept
""").df()
print("\nQ2 — my slice, one row per page-month after each contract filter")
print(slice_counts.to_string(index=False))
receipts["q2_rows_in_month"] = int(counts.rows_in_month[0])
receipts["q2_pages_in_slice"] = int(slice_counts.pages_in_my_slice[0])
receipts["q2_clients_in_slice"] = int(slice_counts.clients_in_my_slice[0])

# --- Q3: availability is three-valued -> filter with IS TRUE -----------------
avail = con.sql(f"""
    SELECT COUNT(*)                                        AS rows_total,
           COUNT(*) FILTER (gsc_data_available IS TRUE)    AS gsc_is_true,
           COUNT(*) FILTER (gsc_data_available IS FALSE)   AS gsc_is_false,
           COUNT(*) FILTER (gsc_data_available IS NULL)    AS gsc_is_null,
           COUNT(*) FILTER (ga4_data_available IS TRUE)    AS ga4_is_true,
           COUNT(*) FILTER (ga4_data_available IS FALSE)   AS ga4_is_false,
           COUNT(*) FILTER (ga4_data_available IS NULL)    AS ga4_is_null
    FROM {fact(FEATURE_MONTH)}
""").df()
print("\nQ3 — availability flags in the feature month")
print(avail.to_string(index=False))
surv = avail.gsc_is_true[0] / avail.rows_total[0]
print(f"\nRows surviving `gsc_data_available IS TRUE`: {avail.gsc_is_true[0]:,} of "
      f"{avail.rows_total[0]:,} ({surv:.1%})")
print(f"GA4 rows that are NULL, not FALSE: {avail.ga4_is_null[0]:,} "
      f"({avail.ga4_is_null[0] / avail.rows_total[0]:.1%}) — `= FALSE` would silently drop these, "
      f"which is why every filter above uses IS TRUE / IS NOT TRUE.")
receipts["q3_gsc_is_true_share"] = round(float(surv), 4)
receipts["q3_ga4_null_share"] = round(float(avail.ga4_is_null[0] / avail.rows_total[0]), 4)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Q1 — duplicate (date x client x content) rows found: 0  (0 = the fact grain is what I said it is)

Q2 — the whole feature month
 rows_in_month  content_items  clients  first_day   last_day
       9841378         331437       55 2026-03-01 2026-03-31

Q2 — my slice, one row per page-month after each contract filter
 pages_after_availability_and_history  pages_in_my_slice  clients_in_my_slice
                               145238              85453                   27

Q3 — availability flags in the feature month
 rows_total  gsc_is_true  gsc_is_false  gsc_is_null  ga4_is_true  ga4_is_false  ga4_is_null
    9841378      3611061       6230317            0       413966       6408671      3018741

Rows surviving `gsc_data_available IS TRUE`: 3,611,061 of 9,841,378 (36.7%)
GA4 rows that are NULL, not FALSE: 3,018,741 (30.7%) — `= FALSE` would silently drop these, which is why every filter above uses IS TRUE / IS NOT TRUE.


### 3b. The five-feature frame (built from the same month)

One row per page-month, five features, all measured inside `month=2026-03` — then the label
joined on from `month=2026-04`. The missingness check follows the frame: `momentum_in_month` is
undefined for pages whose first-half March impressions are zero, and that gap is **not random**
(it marks pages that only started earning impressions mid-month), so it gets a `has_` flag
instead of a silent `fillna`.

In [12]:
features_df = con.sql(f"""
    WITH march AS (
        SELECT f.client_hash_id,
               f.content_hash_id,
               SUM(f.gsc_impressions)                                   AS imp_mar,
               SUM(f.gsc_clicks)                                        AS clk_mar,
               COUNT(*) FILTER (f.gsc_impressions > 0)                  AS active_days,
               SUM(f.gsc_sum_position)                                  AS sum_pos_mar,
               SUM(f.gsc_impressions) FILTER (f.report_date >= DATE '2026-03-17') AS imp_late,
               SUM(f.gsc_impressions) FILTER (f.report_date <  DATE '2026-03-17') AS imp_early
        FROM {fact(FEATURE_MONTH)} f
        JOIN {DIM_CLIENTS} c USING (client_hash_id)
        WHERE f.gsc_data_available IS TRUE
          AND c.gsc_data_start <= DATE '{MIN_HISTORY}'
        GROUP BY 1, 2
        HAVING SUM(f.gsc_impressions) >= {MIN_IMP_MAR}
    )
    SELECT client_hash_id, content_hash_id, imp_mar, active_days,
           sum_pos_mar / NULLIF(imp_mar, 0) AS pos_mar,
           clk_mar     / NULLIF(imp_mar, 0) AS ctr_mar,
           imp_late    / NULLIF(imp_early, 0) AS momentum_in_month
    FROM march
""").df()

label_df = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr
    FROM {fact(LABEL_MONTH)}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1
""").df()

# A1 claims content_hash_id alone identifies a row inside one scoring month — so it is also a
# safe merge key. Check before merging: a duplicated key would silently fan the frame out.
assert not features_df.content_hash_id.duplicated().any(), "content_hash_id is not unique in-month"
assert not label_df.content_hash_id.duplicated().any(), "label key is not unique"
print(f"merge key check — {features_df.content_hash_id.nunique():,} unique content ids "
      f"in {len(features_df):,} feature rows (1:1, so the join cannot fan out)")

frame = features_df.merge(label_df, on="content_hash_id", how="left")
assert len(frame) == len(features_df), "the label join changed the row count"
frame["imp_apr"] = frame["imp_apr"].fillna(0)   # no April rows = the page earned no impressions
frame["label_declining"] = (frame.imp_apr < (1 - DECLINE_DROP) * frame.imp_mar).astype(int)

# Missingness, handled on purpose (never a blind fillna)
print("null share per built column:")
print(frame[["imp_mar", "active_days", "pos_mar", "ctr_mar", "momentum_in_month"]].isna().mean().round(4).to_string())
frame["has_first_half_traffic"] = frame.momentum_in_month.notna().astype(int)
frame["momentum_in_month"] = frame.momentum_in_month.fillna(1.0)   # 1.0 = "flat", flagged above
frame["log_imp_mar"] = np.log1p(frame.imp_mar)

print(f"\nframe: {len(frame):,} page-month rows | {frame.client_hash_id.nunique()} clients")
print(f"label base rate (share declining next month): {frame.label_declining.mean():.2%}")
print(f"pages with no first-half March traffic (flagged, not dropped): "
      f"{(1 - frame.has_first_half_traffic).sum():,} ({1 - frame.has_first_half_traffic.mean():.2%})")
print("\nfeature summary (no ids printed — they are pseudonyms and stay out of the output):")
print(frame[FEATURES].describe().T[["count", "mean", "50%", "max"]].to_string())

receipts["frame_rows"] = int(len(frame))
receipts["frame_clients"] = int(frame.client_hash_id.nunique())
receipts["label_base_rate"] = round(float(frame.label_declining.mean()), 4)
receipts["momentum_missing_share"] = round(float(1 - frame.has_first_half_traffic.mean()), 4)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

merge key check — 85,453 unique content ids in 85,453 feature rows (1:1, so the join cannot fan out)
null share per built column:
imp_mar              0.0000
active_days          0.0000
pos_mar              0.0000
ctr_mar              0.0000
momentum_in_month    0.0281

frame: 85,453 page-month rows | 27 clients
label base rate (share declining next month): 53.78%
pages with no first-half March traffic (flagged, not dropped): 2,397 (2.81%)

feature summary (no ids printed — they are pseudonyms and stay out of the output):
                     count       mean        50%          max
log_imp_mar        85453.0   6.837644   6.689599    13.332827
active_days        85453.0  28.870291  31.000000    31.000000
pos_mar            85453.0  14.685329   8.307325   106.890995
ctr_mar            85453.0   0.002411   0.001145     0.155844
momentum_in_month  85453.0   3.695070   1.000000  4091.000000


### 3c. The trap — one label-derived column, on purpose

w02's lesson, run on real warehouse data. I add **one** column that could not possibly be known
on 2026-03-31 — `imp_apr`, the April impression total the label is computed from — score the
model, then delete it and keep the honest number. The split is **grouped by client**
(`GroupShuffleSplit`, seed 42): pages of a client are either all in train or all in test, because
a random row split would let the model see the same client's pages on both sides.

In [13]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
train_idx, test_idx = next(splitter.split(frame, groups=frame.client_hash_id))
train, test = frame.iloc[train_idx], frame.iloc[test_idx]
print(f"grouped split — train {len(train):,} rows / {train.client_hash_id.nunique()} clients | "
      f"test {len(test):,} rows / {test.client_hash_id.nunique()} clients")
print(f"test-set base rate (what picking {TOP_K} pages at random would score): "
      f"{test.label_declining.mean():.2%}")


def quick_score(cols, tag):
    """Fit, rank the test set, report the lane metric (precision@50) and AUC."""
    model = HistGradientBoostingClassifier(random_state=SEED, max_iter=150)
    model.fit(train[cols], train.label_declining)
    proba = model.predict_proba(test[cols])[:, 1]
    queue = test.assign(risk=proba).nlargest(TOP_K, "risk")
    p_at_k, auc = queue.label_declining.mean(), roc_auc_score(test.label_declining, proba)
    print(f"{tag:<34s} precision@{TOP_K} = {p_at_k:.2%}   AUC = {auc:.3f}")
    return {"precision_at_k": round(float(p_at_k), 4), "auc": round(float(auc), 4)}


honest = quick_score(FEATURES, "honest 5 features")
leaked = quick_score(FEATURES + ["imp_apr"], "+ imp_apr (deliberate leak)")

# ...and now delete it. The leaked column never returns to FEATURES.
assert "imp_apr" not in FEATURES
print(f"\nLeak deleted. The number I keep is the honest one: "
      f"precision@{TOP_K} = {honest['precision_at_k']:.2%}, AUC = {honest['auc']:.3f}.")
print(f"The leak bought +{(leaked['auc'] - honest['auc']):.3f} AUC by reading April — "
      f"a perfect score that would score nothing on 2026-03-31, when April does not exist yet.")

receipts["baseline_random_top50"] = round(float(test.label_declining.mean()), 4)
receipts["honest_score"] = honest
receipts["leaked_score"] = leaked

grouped split — train 59,682 rows / 18 clients | test 25,771 rows / 9 clients
test-set base rate (what picking 50 pages at random would score): 49.36%
honest 5 features                  precision@50 = 94.00%   AUC = 0.665
+ imp_apr (deliberate leak)        precision@50 = 100.00%   AUC = 1.000

Leak deleted. The number I keep is the honest one: precision@50 = 94.00%, AUC = 0.665.
The leak bought +0.334 AUC by reading April — a perfect score that would score nothing on 2026-03-31, when April does not exist yet.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**The named limitation of this slice: it is one month pair.** Every label in this contract comes
from a single March → April transition, so a page's "decline" and *anything that moved the whole
web in April 2026* — a ranking update, a seasonal dip, a tracking change — are the same
observation. This data can therefore support a **directional, decision-support** claim ("these
pages are the ones most likely to keep bleeding, work them first") and can never support a causal
one ("this page declined *because* of X"). Fixing it means repeating the contract on several
month pairs and checking the score holds; that is validation work (ML-09), not a query I can
write today.

Three more limits the queries below measure, each of which narrows what the number in section 3c means:

1. **The panel is unbalanced and concentrated.** Of the clients in the release, only some have
   GSC history reaching back before 2026-01-01, and my slice's pages are dominated by a handful
   of them — so a cross-client split has few effective groups, and one atypical client can move
   the metric. This is why the split is grouped, and why I read precision@50 as directional.
2. **GA4 is effectively absent here, so "engagement" is out of reach.** With ~4% of March rows
   flagged `ga4_data_available IS TRUE`, this slice cannot say anything about whether readers
   *engaged* with a page — only about what search showed and clicked. Any engagement-flavoured
   conclusion would be a guess dressed as a measurement.
3. **A page missing from April is not the same as a page that died.** I fill missing April
   impressions with 0, which reads "no impressions". A page unpublished, redirected, or dropped
   from tracking in April looks identical in this data to a page that simply lost all its
   traffic — the release ships no unpublish/redirect event to tell them apart.

In [14]:
# 1. How concentrated is my slice across clients? (ids withheld — pseudonyms stay out of output)
share = frame.client_hash_id.value_counts(normalize=True)
print("client concentration in my slice (ranked, ids withheld):")
for rank, pct in enumerate(share.head(5).values, start=1):
    print(f"  client #{rank}: {pct:.1%} of page-month rows")
print(f"  top 3 clients combined: {share.head(3).sum():.1%} of {len(frame):,} rows "
      f"across {frame.client_hash_id.nunique()} clients")

# 2. GA4 coverage in the feature month (measured in Q3) — restated as the limit it is
print(f"\nGA4 availability in {FEATURE_MONTH}: ga4_data_available IS TRUE on "
      f"{avail.ga4_is_true[0]:,} of {avail.rows_total[0]:,} rows "
      f"({avail.ga4_is_true[0] / avail.rows_total[0]:.1%}); NULL on "
      f"{receipts['q3_ga4_null_share']:.1%} — engagement questions are out of scope for this slice.")

# 3. Pages present in March but with no April rows at all — 'gone' vs 'quiet' is unknowable here
missing_apr = (~frame.content_hash_id.isin(label_df.content_hash_id)).sum()
print(f"\nPages in my slice with no April rows at all: {missing_apr:,} "
      f"({missing_apr / len(frame):.1%}) — filled as 0 impressions, but unpublished and "
      f"collapsed look identical in this release.")

# Patterned missingness lives in dim_content too — one join, read-only, no features taken from it
ct = con.sql(f"""
    SELECT content_type, COUNT(*) AS items,
           AVG(CASE WHEN word_count    IS NULL THEN 1.0 ELSE 0 END) AS word_count_null,
           AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END) AS search_volume_null
    FROM {DIM_CONTENT}
    GROUP BY 1 ORDER BY items DESC
""").df()
print("\ndim_content — missingness follows content_type (one type is 100% missing search_volume).")
print("Read as: if I ever add a dim_content feature, it must come with a has_ flag, not a fillna.")
print(ct.to_string(index=False))

receipts["top3_client_share"] = round(float(share.head(3).sum()), 4)
receipts["pages_missing_april"] = int(missing_apr)

out_dir = pathlib.Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "w03_data_contract_receipts.json").write_text(json.dumps(receipts, indent=2), encoding="utf-8")
print(f"\nreceipts written to work/outputs/w03_data_contract_receipts.json ({len(receipts)} entries)")
print(json.dumps(receipts, indent=2))

client concentration in my slice (ranked, ids withheld):
  client #1: 25.3% of page-month rows
  client #2: 20.3% of page-month rows
  client #3: 14.9% of page-month rows
  client #4: 9.9% of page-month rows
  client #5: 9.4% of page-month rows
  top 3 clients combined: 60.5% of 85,453 rows across 27 clients

GA4 availability in 2026-03: ga4_data_available IS TRUE on 413,966 of 9,841,378 rows (4.2%); NULL on 30.7% — engagement questions are out of scope for this slice.

Pages in my slice with no April rows at all: 380 (0.4%) — filled as 0 impressions, but unpublished and collapsed look identical in this release.

dim_content — missingness follows content_type (one type is 100% missing search_volume).
Read as: if I ever add a dim_content feature, it must come with a has_ flag, not a fillna.
      content_type  items  word_count_null  search_volume_null
   keyword article 459174         0.380760            0.186411
    feedly article  57024         0.051382            1.000000
comparison